<a href="https://colab.research.google.com/github/Andrescolonia/S9_Taller_Visualizacion_Datos/blob/main/S9_Taller_Visualizacion_Datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller de Visualización de Datos con Python
## Análisis de dataset: Novel Corona Virus 2019

**Objetivos:**
- Identificar y aplicar técnicas de visualización adecuadas según el tipo de dato (categórico, continuo, temporal).

- Analizar distribuciones de datos mediante histogramas, polígonos de frecuencia y diagramas de caja.

- Crear visualizaciones avanzadas (mapas de calor geográficos) para comunicar hallazgos complejos.

**Instrucciones:**
- Trabajen en grupos de 3 estudiantes
- Tiempo disponible: 2 horas
- Entreguen un notebook de Jupyter con el código y respuestas a las preguntas
- Incluyan comentarios explicando su proceso de análisis

**Dataset:** `covid_19_data.csv` - Dataset global de COVID-19 que incluye casos confirmados, muertes y recuperaciones por país y fecha, desde enero de 2020

---

## Actividad 1: Carga y Exploración Inicial del Dataset (15 minutos)

1.	Cargar el dataset COVID-19 Global.

- Navegar a https://www.kaggle.com/datasets/sudalairajkumar/novel-corona-virus-2019-dataset/data
- Revisar la descripción de las columnas del dataset
- Dar click en el botón: '<> Code' para desplegar el código necesario para importar el dataset.
- Copiar el código y pegarlo en el Jupiter Notebook.
- Modifique la variable file_path así: file_path="covid_19_data.csv"
- Ejecute el código y tendrá el dataset importado en la variable df

2.	Realizar una exploración inicial para entender la estructura del dataset.
3.	Preparar los datos para el análisis (conversión de tipos, manejo de valores nulos, etc.).

---

## Actividad 2: Visualizaciones Básicas (35 minutos)

1.	Gráfico de Barras:
*	Representen los 10 países con mayor número de casos confirmados.
*	Creen un gráfico de barras comparando muertes y recuperados para los 5 países más afectados.
* ¿Qué pueden concluir acerca de los países más afectados y los datos reportados acerca de muertes y recuperados?

- NOTA: Tenga en cuenta que las columnas Confirmed, Deaths, Recovered son acumulativas, es decir que en cada dato que aparece se consolidan los datos de los días anteriores. También es normal que los valores máximos de casos Confirmados no coincidan con los valores máximos de Muertes o Recuperados

2.	Gráfico de Líneas:
*	Visualicen la evolución temporal de casos confirmados, muertes y recuperados a nivel global.
* ¿Qué pueden observar acerca del comportamiento de los datos?

3.	Histograma y Polígono de Frecuencia:
*	Creen un histograma de la distribución de casos confirmados en Estados Unidos.
*	Generen tres polígonos de frecuencia, superpuestos, mostrando la distribución de casos confirmados para Francia, Brasil y Turquía.
*	¿Qué pueden concluir acerca de las distribuciones?

---

## ACTIVIDAD 3: Visualizaciones Estadísticas (35 minutos)

1.	Diagrama de Dispersión:
*	Crear un diagrama de dispersión entre casos confirmados y muertes. (No incluya países con menos de 1000 casos confirmados)
*	Calcular la correlación entre ambas variables.
* ¿Cómo se relacionan las variables?

2.	Diagrama de Caja y Bigotes:
*	Cree diagramas de cajas para la distribución de casos confirmados en los países sudamericanos
* ¿Qué países presentan valores atípicos?¿Qué puede concluir al respecto y acerca de los datos?
* NOTA: Use escala logarítmica para visualizar mejor

3.	Diagrama Cuantil-Cuantil:
*	Creen el diagrama Cuantil-Cuantil utilizando los Casos confirmados.
* Creen otro diagrama Cuantil-Cuantil en escala logarítmica para Casos confirmados.
* ¿Cómo se comportan los datos reales en comparación con las líneas teóricas?
* NOTA: Utilicen la función probplot de la librería scipy.stats

---

## ACTIVIDAD 4: Análisis Visual Avanzado (35 minutos)

1.	Mapa de Calor de Correlación:
*	Crear un mapa de calor de correlación entre todas las variables numéricas del dataset.
* ¿Cuál es la correlación más fuerte?

2. Mapa de Calor Geográfico:
*	Crear un mapa de calor geográfico mostrando cantidad de muertes por país
* ¿Qué puede concluir acerca del gráfico?
* NOTA: Utilice la librería plotly.express, función cloropeth. Recuerde agrupar los datos por fecha y país, y sumar las otras categorías para hallar los datos reales.

---

## Discusión y Conclusiones (Tiempo restante)

Basándose en todas las visualizaciones generadas, escriban las 3 conclusiones más importantes que se pueden extraer de los datos.

---

**Entrega:**
- Notebook de Jupyter con: Nombres de integrantes, código, resultados, respuestas

In [ ]:
# Carga de dataset y librerías base
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy.stats import probplot

sns.set_theme(style="whitegrid")

# Intentar cargar primero un archivo local para facilitar reproducibilidad
local_file = Path("covid_19_data.csv")

if local_file.exists():
    df = pd.read_csv(local_file)
    print(f"Dataset cargado localmente desde: {local_file.resolve()}")
else:
    import kagglehub
    from kagglehub import KaggleDatasetAdapter

    file_path = "covid_19_data.csv"
    df = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "sudalairajkumar/novel-corona-virus-2019-dataset",
        file_path
    )
    print("Dataset cargado desde KaggleHub")

print(df.head())


## Exploracion de datos y preparacion para analisis

In [ ]:
# 1) Exploración inicial
print("Dimensiones:", df.shape)
print("\nColumnas:")
print(df.columns.tolist())

print("\nInformación general:")
df.info()

print("\nValores nulos por columna:")
print(df.isnull().sum())

print("\nDuplicados exactos:", df.duplicated().sum())

# 2) Copia para limpieza
df_clean = df.copy()

# 3) Normalizar nombres de columnas
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace("/", "_", regex=False)
    .str.replace(" ", "_", regex=False)
)

# 4) Conversión de tipos
if "observationdate" in df_clean.columns:
    df_clean["observationdate"] = pd.to_datetime(df_clean["observationdate"], errors="coerce")

if "last_update" in df_clean.columns:
    df_clean["last_update"] = pd.to_datetime(df_clean["last_update"], errors="coerce")

for col in ["confirmed", "deaths", "recovered"]:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# 5) Manejo de nulos
if "province_state" in df_clean.columns:
    df_clean["province_state"] = df_clean["province_state"].fillna("Sin dato")
if "country_region" in df_clean.columns:
    df_clean["country_region"] = df_clean["country_region"].fillna("Sin dato")

for col in ["confirmed", "deaths", "recovered"]:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(0)

if "observationdate" in df_clean.columns:
    df_clean = df_clean.dropna(subset=["observationdate"])

# 6) Eliminar duplicados
df_clean = df_clean.drop_duplicates()

print("\n--- DATASET LIMPIO ---")
print("Dimensiones finales:", df_clean.shape)
print("Rango de fechas:", df_clean["observationdate"].min(), "->", df_clean["observationdate"].max())
print("\nPrimeras filas limpias:")
print(df_clean.head())


## Preparacion de datos para visualizacion

In [ ]:
# Preparación para evitar doble conteo país/provincia
df_fix = df.copy()
df_fix["ObservationDate"] = pd.to_datetime(df_fix["ObservationDate"], errors="coerce")

# Detectar mezcla entre fila nacional y filas subnacionales
prov = df_fix["Province/State"].fillna("").str.strip()

has_subnational = (
    prov.ne("")
    .groupby([df_fix["ObservationDate"], df_fix["Country/Region"]])
    .transform("any")
)

has_national = (
    prov.eq("")
    .groupby([df_fix["ObservationDate"], df_fix["Country/Region"]])
    .transform("any")
)

mixed_structure = has_subnational & has_national
drop_mask = mixed_structure & prov.eq("")
df_fix_clean = df_fix.loc[~drop_mask].copy()

print("Filas originales:", len(df_fix))
print("Filas eliminadas por posible doble conteo:", int(drop_mask.sum()))
print("Filas finales:", len(df_fix_clean))

country_daily_clean = (
    df_fix_clean
    .groupby(["ObservationDate", "Country/Region"], as_index=False)[["Confirmed", "Deaths", "Recovered"]]
    .sum()
    .sort_values(["ObservationDate", "Confirmed"], ascending=[True, False])
)

print(country_daily_clean.head())


## a) Representar los 10 países con mayor número de casos confirmados

In [ ]:
latest_date = country_daily_clean["ObservationDate"].max()

latest_country_clean = country_daily_clean[
    country_daily_clean["ObservationDate"] == latest_date
].copy()

top10_confirmed_clean = (
    latest_country_clean
    .sort_values("Confirmed", ascending=False)
    .head(10)
)

print(top10_confirmed_clean[["Country/Region", "Confirmed"]])

plt.figure(figsize=(12, 6))
plt.bar(top10_confirmed_clean["Country/Region"], top10_confirmed_clean["Confirmed"])
plt.title("Top 10 países con mayor número de casos confirmados")
plt.xlabel("País")
plt.ylabel("Casos confirmados acumulados")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## b) Comparar muertes y recuperados para los 5 países más afectados

In [ ]:
# Tomar los 5 países con más confirmados en la última fecha disponible
top5_affected_clean = top10_confirmed_clean.head(5).copy()

top5_plot = top5_affected_clean[["Country/Region", "Deaths", "Recovered"]].copy()
top5_plot["Deaths_M"] = top5_plot["Deaths"] / 1_000_000
top5_plot["Recovered_M"] = top5_plot["Recovered"] / 1_000_000

print(top5_plot[["Country/Region", "Deaths", "Recovered"]])

x = np.arange(len(top5_plot))
width = 0.35

plt.figure(figsize=(10, 6))
plt.bar(x - width/2, top5_plot["Deaths_M"], width, label="Muertes")
plt.bar(x + width/2, top5_plot["Recovered_M"], width, label="Recuperados")

plt.title("Muertes y recuperados en los 5 países más afectados")
plt.xlabel("País")
plt.ylabel("Casos (millones)")
plt.xticks(x, top5_plot["Country/Region"], rotation=45)
plt.legend()
plt.tight_layout()
plt.show()


## 3. Gráfico de Líneas

In [ ]:
global_daily_clean = (
    country_daily_clean
    .groupby("ObservationDate", as_index=False)[["Confirmed", "Deaths", "Recovered"]]
    .sum()
    .sort_values("ObservationDate")
)

plt.figure(figsize=(12, 6))
plt.plot(global_daily_clean["ObservationDate"], global_daily_clean["Confirmed"], label="Confirmados")
plt.plot(global_daily_clean["ObservationDate"], global_daily_clean["Deaths"], label="Muertes")
plt.plot(global_daily_clean["ObservationDate"], global_daily_clean["Recovered"], label="Recuperados")

plt.title("Evolución temporal global de casos confirmados, muertes y recuperados")
plt.xlabel("Fecha")
plt.ylabel("Número de casos acumulados")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Histograma

In [ ]:
us_data_clean = country_daily_clean[
    country_daily_clean["Country/Region"] == "US"
]["Confirmed"]

print(us_data_clean.describe())

plt.figure(figsize=(10, 6))
plt.hist(us_data_clean, bins=20)
plt.title("Histograma de casos confirmados en Estados Unidos")
plt.xlabel("Casos confirmados acumulados")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()

## 5. Polígonos de Frecuencia

In [ ]:
france_data_clean = country_daily_clean[country_daily_clean["Country/Region"] == "France"]["Confirmed"]
brazil_data_clean = country_daily_clean[country_daily_clean["Country/Region"] == "Brazil"]["Confirmed"]
turkey_data_clean = country_daily_clean[country_daily_clean["Country/Region"] == "Turkey"]["Confirmed"]

all_data = pd.concat([france_data_clean, brazil_data_clean, turkey_data_clean])
bins = np.histogram_bin_edges(all_data, bins=15)

fr_counts, _ = np.histogram(france_data_clean, bins=bins)
br_counts, _ = np.histogram(brazil_data_clean, bins=bins)
tr_counts, _ = np.histogram(turkey_data_clean, bins=bins)

bin_centers = (bins[:-1] + bins[1:]) / 2

plt.figure(figsize=(10, 6))
plt.plot(bin_centers, fr_counts, marker="o", label="France")
plt.plot(bin_centers, br_counts, marker="o", label="Brazil")
plt.plot(bin_centers, tr_counts, marker="o", label="Turkey")

plt.title("Polígonos de frecuencia de casos confirmados")
plt.xlabel("Casos confirmados acumulados")
plt.ylabel("Frecuencia")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
latest_date = country_daily_clean["ObservationDate"].max()

latest_country_clean = country_daily_clean[
    country_daily_clean["ObservationDate"] == latest_date
].copy()

print("Última fecha disponible en el dataset:", latest_date.date())
print(latest_country_clean.head())


## 1) Diagrama de Dispersión

### a) Confirmados vs Muertes

In [ ]:
scatter_data = latest_country_clean[
    latest_country_clean["Confirmed"] >= 1000
].copy()

print(scatter_data[["Country/Region", "Confirmed", "Deaths"]].head())
print("Número de países analizados:", len(scatter_data))

plt.figure(figsize=(10, 6))
plt.scatter(scatter_data["Confirmed"], scatter_data["Deaths"], alpha=0.7)
plt.xscale("log")
plt.yscale("log")
plt.title("Confirmados vs muertes (escala logarítmica)")
plt.xlabel("Casos confirmados")
plt.ylabel("Muertes")
plt.tight_layout()
plt.show()

### b) Correlación

In [ ]:
corr = scatter_data["Confirmed"].corr(scatter_data["Deaths"], method="pearson")
print(f"Correlación de Pearson: {corr:.4f}")

## 2) Diagrama de Caja y Bigotes

### a) Filtrar países sudamericanos

In [ ]:
south_america = [
    "Argentina", "Bolivia", "Brazil", "Chile", "Colombia", "Ecuador",
    "Guyana", "Paraguay", "Peru", "Suriname", "Uruguay", "Venezuela",
    "French Guiana"
]

sa_data = country_daily_clean[
    country_daily_clean["Country/Region"].isin(south_america)
].copy()

countries_present = sorted(sa_data["Country/Region"].unique())
print("Países encontrados:", countries_present)

### b) Crear boxplot en escala logarítmica

In [ ]:
boxplot_data = [
    sa_data.loc[sa_data["Country/Region"] == country, "Confirmed"].values
    for country in countries_present
]

plt.figure(figsize=(14, 7))
plt.boxplot(boxplot_data, labels=countries_present, showfliers=True)
plt.yscale("log")
plt.title("Distribución de casos confirmados en países sudamericanos")
plt.xlabel("País")
plt.ylabel("Casos confirmados (escala log)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### c) Identificar valores atípicos por país

In [ ]:
outlier_summary = []

for country in countries_present:
    vals = sa_data.loc[sa_data["Country/Region"] == country, "Confirmed"]

    q1 = vals.quantile(0.25)
    q3 = vals.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = vals[(vals < lower) | (vals > upper)]

    outlier_summary.append({
        "Country/Region": country,
        "Outliers": int(outliers.count()),
        "MaxConfirmed": vals.max()
    })

outlier_df = pd.DataFrame(outlier_summary).sort_values(
    by=["Outliers", "MaxConfirmed"], ascending=False
)

print(outlier_df)

## 3) Diagrama Cuantil-Cuantil

### a) Q-Q plot con Confirmed

In [ ]:
qq_data = latest_country_clean["Confirmed"].dropna()

plt.figure(figsize=(8, 6))
probplot(qq_data, dist="norm", plot=plt)
plt.title("Diagrama Q-Q de casos confirmados")
plt.tight_layout()
plt.show()

### b) Q-Q plot en escala logarítmica

In [ ]:
qq_log_data = latest_country_clean.loc[
    latest_country_clean["Confirmed"] > 0, "Confirmed"
]

log_confirmed = np.log10(qq_log_data)

plt.figure(figsize=(8, 6))
probplot(log_confirmed, dist="norm", plot=plt)
plt.title("Diagrama Q-Q de log10(casos confirmados)")
plt.tight_layout()
plt.show()

In [ ]:


# =========================
# 1. MAPA DE CALOR DE CORRELACIÓN
# =========================
corr_data = country_daily_clean[["Confirmed", "Deaths", "Recovered"]].copy()
corr_matrix = corr_data.corr(method="pearson")

print("Matriz de correlación:")
print(corr_matrix)

fig = px.imshow(
    corr_matrix,
    text_auto=True,
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Mapa de calor de correlación"
)

fig.update_layout(width=700, height=600)
fig.show()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
corr_pairs = corr_matrix.where(mask).stack().sort_values(key=np.abs, ascending=False)

strongest_pair = corr_pairs.index[0]
strongest_value = corr_pairs.iloc[0]

print("Correlación más fuerte:")
print(f"{strongest_pair[0]} vs {strongest_pair[1]} = {strongest_value:.4f}")

# =========================
# 2. MAPA DE CALOR GEOGRÁFICO
# =========================
latest_date = country_daily_clean["ObservationDate"].max()

latest_country_clean = country_daily_clean[
    country_daily_clean["ObservationDate"] == latest_date
].copy()

geo_data = latest_country_clean[["Country/Region", "Deaths"]].copy()

country_map = {
    "US": "United States",
    "UK": "United Kingdom",
    "Korea, South": "South Korea",
    "Taiwan*": "Taiwan",
    "Mainland China": "China",
    "Congo (Kinshasa)": "Democratic Republic of the Congo",
    "Congo (Brazzaville)": "Republic of the Congo",
    "West Bank and Gaza": "Palestine",
    "Burma": "Myanmar",
    "Holy See": "Vatican City",
    "Diamond Princess": np.nan,
    "MS Zaandam": np.nan
}

geo_data["country_plotly"] = geo_data["Country/Region"].replace(country_map)
geo_data["country_plotly"] = geo_data["country_plotly"].fillna(geo_data["Country/Region"])
geo_data = geo_data[~geo_data["country_plotly"].isin(["Diamond Princess", "MS Zaandam"])]

fig = px.choropleth(
    geo_data,
    locations="country_plotly",
    locationmode="country names",
    color="Deaths",
    hover_name="Country/Region",
    color_continuous_scale="Reds",
    title=f"Mapa de calor geográfico de muertes por país ({latest_date.date()})"
)

fig.update_layout(width=1000, height=600)
fig.show()